<div class="dark-title" style="background:linear-gradient(90deg,#1e3a8a,#5b21b6); color:#f1f5f9; padding:20px 32px; border-radius:8px; width:97%;"><style>.dark-title strong{color:#fde047;}</style>

# 💍 婚礼策划 Agent —— Module 2 综合练习

**一句话定位**：把 2.1~2.3 讲的 MCP、runtime context、state、multi-agent 全部串起来，做一个真实能跑的婚礼策划系统。



</div>

本 notebook 把 2.1–2.3 学到的所有东西汇总起来：

| 概念 | 用在哪 |
|---|---|
| 🔌 MCP (stdio) | Venue agent 通过本地 MCP server 调用 Tavily 搜索 |
| 🌐 MCP (streamable_http) | Transport agent 通过远程 MCP 调用 Kiwi 航班查询 |
| 📐 Runtime context (`context_schema`) | 所有 agent 都声明 `WeddingContext` 作为自己的 schema |
| 📦 Runtime context (`context=`) | 调用时传入真实的新人数据 |
| 🗂️ State (`state_schema`) | `WeddingPlanState` 在各个 subagent 之间累积结果 |
| ✏️ State write | DJ agent 通过 `Command(update={...})` 写入歌单 |
| 👁️ State read | Coordinator 通过 `runtime.state` 读取进度 |
| 🤝 Multi-agent | Coordinator 把任务分派给 4 个专精 subagent |
| 💾 Checkpointer | `InMemorySaver` 支持多轮对话 |

---

## 🗂️ 文件结构

```
2.4_5_wedding_exercisee/
├── context_and_state.py   # WeddingContext (dataclass) + WeddingPlanState (AgentState)
├── mcp_servers.py         # FastMCP server (stdio) + get_mcp_tools() 辅助函数
├── subagents.py           # 4 个专精 agent：venue、catering、transport、DJ
├── wedding_planner.py     # Coordinator + 主入口
└── wedding_planner.ipynb  # 本 notebook
```

In [ ]:
import sys
import os
import asyncio

# LESSON (from 2.1_mcp.ipynb): Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__

# Make sure imports from this folder work
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

from dotenv import load_dotenv
load_dotenv('../../../.env')  # load OPENAI_API_KEY, TAVILY_API_KEY

## 步骤 1 —— Context 与 State 的 schema

在创建任何 agent 之前，先定义两个 schema：
- `WeddingContext` —— 一个 dataclass，装新人信息（只读，调用时注入）
- `WeddingPlanState` —— 继承 `AgentState`，新增的字段由 coordinator 在各 subagent 汇报结果后逐步填入

In [ ]:
from context_and_state import WeddingContext, WeddingPlanState

# This is the runtime context — the actual data we'll pass at invoke time
wedding_context = WeddingContext(
    couple_names="Alice & Bob",
    wedding_date="2026-09-15",
    wedding_location="Paris, France",
    guest_count=80,
    budget_usd=50_000,
    music_style="jazz and soul",
    dietary_restrictions="vegan options required",
    home_city="New York",
)

print(wedding_context)

## 步骤 2 —— 连接 MCP server

这里连接两个 MCP server：
1. **本地（stdio）**—— `mcp_servers.py` 作为子进程运行，通过 Tavily 暴露 `search_web`
2. **远程（streamable_http）**—— Kiwi.com 的 MCP server，暴露 `search-flight`

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

`MultiServerMCPClient` 同时支持这两种 transport，返回统一的工具列表——对 agent 来说，调用本地脚本的工具和调用远程公网 server 的工具没有区别。

</div>

In [ ]:
from mcp_servers import get_mcp_tools

mcp_tools, mcp_client = await get_mcp_tools()

print("MCP tools available:")
for t in mcp_tools:
    print(f"  - {t.name}: {t.description[:60]}...")

### 彩蛋：MCP 的 Resources & Prompts

回顾 2.1_mcp 里讲过的：MCP server 暴露的不只是 tools。
- `get_resources()` —— server 提供的静态数据
- `get_prompt()` —— server 定义的可复用 prompt 模板

2.1_mcp 课程里就是这样从 MCP server 直接加载 system prompt 的。

In [ ]:
from mcp_servers import get_mcp_resources_and_prompts

resources, prompt_text = await get_mcp_resources_and_prompts(mcp_client)

print("Resources:", resources)
print("\nPrompt from MCP server:")
print(prompt_text)

## 步骤 3 —— 创建 4 个专精 subagent

每个 subagent 都是一个独立的 `create_agent()`，各自拥有：
- 自己的工具（有的来自 MCP，有的是普通 `@tool`）
- `context_schema=WeddingContext`，这样它知道自己会收到什么形状的 context
- 一个聚焦在自己领域的 `system_prompt`

In [ ]:
from subagents import (
    create_venue_agent,
    create_catering_agent,
    create_transport_agent,
    create_dj_agent,
)

search_tools = [t for t in mcp_tools if t.name == "search_web"]
flight_tools = [t for t in mcp_tools if "flight" in t.name.lower()]

venue_agent     = create_venue_agent(search_tools)
catering_agent  = create_catering_agent()
transport_agent = create_transport_agent(flight_tools)
dj_agent        = create_dj_agent()

print("✅ All 4 subagents created")

## 步骤 4 —— 单独测试一个 subagent

在跑完整的 coordinator 之前，先单独测一下 catering agent。

<div class="dark-info" style="background:#1e293b; color:#e2e8f0; padding:12px 24px; border-left:4px solid #60a5fa; border-radius:4px; width:97%;"><style>.dark-info strong{color:#fbbf24;}</style>

注意调用时传了 `context=wedding_context`——这才是真实数据。agent 在自己的工具里通过 `ToolRuntime` 读取它。

</div>

In [ ]:
from langchain.messages import HumanMessage
from pprint import pprint

response = catering_agent.invoke(
    {"messages": [HumanMessage(content="Plan catering for our wedding")]},
    context=wedding_context,   # ← runtime context injected here
)

print(response["messages"][-1].content)

## 步骤 5 —— 测试 DJ agent（state 写入演示）

DJ agent 比较特殊：它通过 `Command(update={...})` 把歌单写进 `WeddingPlanState`。

调用结束后，你能在返回的 dict 里看到 `playlist_result` —— 这就是写入的 state 字段。

In [ ]:
dj_response = dj_agent.invoke(
    {"messages": [HumanMessage(content="Create a wedding playlist for us")]},
    {"configurable": {"thread_id": "dj-test-1"}},
    context=wedding_context,
)

print("Last message:", dj_response["messages"][-1].content)
print("\nState field 'playlist_result':")
print(dj_response.get("playlist_result", "(not set)"))

### 彩蛋：直接在 invoke 的 dict 里传 state 字段

回顾 2.2_state 课程：你也可以在 invoke 的 dict 里，跟 `messages` 一起直接传初始 state 值。课程里就是这样设置 `favourite_colour`，不需要 agent 先通过工具调用去"发现"它。

```python
# 课程里的例子：
response = agent.invoke(
    {"messages": [...], "favourite_colour": "green"},
    {"configurable": {"thread_id": "10"}}
)
```

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**易踩坑**：这种「直接预置 state」的写法只对调用方已经提前知道答案的场景有用——它跳过了 agent 自己去调用工具获取数据的过程，适合做测试种子数据，但不能当作常规交互方式依赖。

</div>

In [ ]:
# Pass a pre-set playlist_result directly in the invoke dict
# This seeds the state without the agent needing to call save_playlist_to_state
dj_response_seeded = dj_agent.invoke(
    {
        "messages": [HumanMessage(content="What playlist do I have?")],
        "playlist_result": "Pre-seeded: At Last by Etta James, Fly Me to the Moon by Sinatra"
    },
    {"configurable": {"thread_id": "dj-test-seeded"}},
    context=wedding_context,
)

print("Agent response:", dj_response_seeded["messages"][-1].content)
print("\nState playlist_result:", dj_response_seeded.get("playlist_result", "(not set)"))

## 步骤 6 —— 运行完整的 coordinator

现在跑完整的 multi-agent 系统。

Coordinator 会：
1. 通过包装工具依次调用 4 个 subagent
2. 每个结果都写入 `WeddingPlanState`
3. Coordinator 通过 `read_planning_progress` 读取进度
4. 通过 `write_final_plan` 写入最终拼好的方案

<div class="dark-warning" style="background:#2a2418; color:#fde68a; padding:12px 24px; border-left:4px solid #fbbf24; border-radius:4px; width:97%;"><style>.dark-warning strong{color:#f9a8d4;}</style>

**注意**：这一步会真实调用 Tavily + Kiwi + OpenAI 的 API，可能要等 30–60 秒。

</div>

In [ ]:
from wedding_planner import build_coordinator

coordinator, _ = await build_coordinator(wedding_context)

config = {"configurable": {"thread_id": "wedding-plan-notebook-1"}}

user_request = (
    f"Please plan our complete wedding! "
    f"We are {wedding_context.couple_names}, getting married on {wedding_context.wedding_date} "
    f"in {wedding_context.wedding_location}. "
    f"We have {wedding_context.guest_count} guests and a budget of ${wedding_context.budget_usd:,}. "
    f"We love {wedding_context.music_style} music and need {wedding_context.dietary_restrictions}. "
    f"We're flying from {wedding_context.home_city}. Please coordinate everything!"
)

response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content=user_request)]},
    config=config,
    context=wedding_context,
)

print(response["messages"][-1].content)

## 步骤 7 —— 查看累积的 state

Coordinator 跑完之后，所有 subagent 的结果都在 state 里。

这正是 `WeddingPlanState` 的价值所在——跨轮次之间什么都不会丢。

In [ ]:
print("📊 State after full planning run:")
print("-" * 50)
for field in ["venue_result", "catering_result", "transport_result", "playlist_result", "final_plan"]:
    value = response.get(field, "")
    if value:
        print(f"\n✅ {field}:")
        print(value[:200] + "..." if len(value) > 200 else value)
    else:
        print(f"\n⏳ {field}: (not set)")

## 步骤 8 —— 多轮追问（checkpointer 演示）

因为用了同一个 `thread_id` 的 `InMemorySaver`，coordinator 记得整段对话。可以直接追问，不需要重新跑一遍所有 subagent。

<div class="dark-success" style="background:#1a2e1f; color:#bbf7d0; padding:10px 24px; border-left:4px solid #4ade80; border-radius:4px; width:97%;"><style>.dark-success strong{color:#fbbf24;}</style>

**✨ 一句话带走**：这就是 MCP + runtime context + state + multi-agent + checkpointer 五件套合体后的效果——一个能记住上下文、分工明确、又能对接外部真实数据源的完整 agent 系统。

</div>

In [ ]:
follow_up = "Can you remind me just the playlist and the venue recommendation?"

follow_up_response = await coordinator.ainvoke(
    {"messages": [HumanMessage(content=follow_up)]},
    config=config,   # same thread_id — coordinator remembers everything
    context=wedding_context,
)

print(follow_up_response["messages"][-1].content)